In [30]:
!pip install chromadb pypdf langgraph

In [31]:
from pypdf import PdfReader

reader = PdfReader("data/sample.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

print(text[:300])

Customer Support Knowledge Base
1. Introduction
Welcome to our Customer Support Knowledge Base. This document provides detailed information
about policies, services, and procedures to help customers resolve their queries efficiently. The
system is designed to assist users with common issues such as 


In [32]:
def split_text(text, chunk_size=300):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

chunks = split_text(text)

print("Total chunks:", len(chunks))

Total chunks: 10


In [33]:
import chromadb

client = chromadb.Client()
collection = client.create_collection("rag_collection")

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk],
        ids=[str(i)]
    )

print("Stored successfully")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
C:\Users\sweth\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [02:24<00:00, 576kiB/s] 
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Stored successfully


In [34]:
def retrieve(query):
    results = collection.query(
        query_texts=[query],
        n_results=3
    )
    return results["documents"][0]

In [35]:
def generate_answer(query):
    docs = retrieve(query)
    context = " ".join(docs)

    if len(context.strip()) == 0:
        return "I don't know", context

    return context[:300], context

In [36]:
answer, context = generate_answer("What is refund policy?")
print(answer)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


ather or logistics issues
4. Refund Policy
Customers are eligible for refunds under certain conditions.
 Refund requests must be made within 7 days of delivery
 Refunds are processed within 5–10 business days
 Refunds will be credited to the original payment method
 Shipping charges are non-refu


In [37]:
def check_confidence(answer):
    if "I don't know" in answer:
        return False
    return True

def human_escalation(query):
    return input("Human answer: ")


In [47]:

from typing import TypedDict

class State(TypedDict):
    query: str
    answer: str
    confidence: bool


In [52]:
def process_node(state):
    query = state.get("query", "")   # SAFE ACCESS

    answer, context = generate_answer(query)

    return {
        "query": query,
        "answer": answer,
        "confidence": check_confidence(answer)
    }
def hitl_node(state):
    print("Escalating to human...")

    query = state.get("query", "")

    human_answer = input("Human answer: ")

    return {
        "query": query,
        "answer": human_answer,
        "confidence": True
    }
def output_node(state):
    print("Final Answer:", state["answer"])
    return state


In [53]:
def route(state):
    return "output" if state["confidence"] else "hitl"

In [54]:
graph = StateGraph(State)

graph.add_node("process", process_node)
graph.add_node("hitl", hitl_node)
graph.add_node("output", output_node)

graph.set_entry_point("process")

graph.add_conditional_edges(
    "process",
    route,
    {"output": "output", "hitl": "hitl"}
)

graph.add_edge("hitl", "output")

app = graph.compile()

In [55]:
query = input("Ask your question: ")

result = app.invoke({"query": query})

print(result)

Final Answer: ndable
5. Return Policy
We accept returns if the product meets the following conditions:
 Product must be unused and in original packaging
 Return request should be initiated within 10 days Damaged or defective products are eligible for replacement
6. Cancellation Policy
Customers can cancel orde
{'query': 'what conditions are applied for returning a product?', 'answer': 'ndable\n5. Return Policy\nWe accept returns if the product meets the following conditions:\n\uf0b7 Product must be unused and in original packaging\n\uf0b7 Return request should be initiated within 10 days\uf0b7 Damaged or defective products are eligible for replacement\n6. Cancellation Policy\nCustomers can cancel orde', 'confidence': True}
